# Hiệu chỉnh AI label


In [1]:
from __future__ import annotations

import re
import unicodedata
from pathlib import Path
from typing import Iterable, List

import pandas as pd


# ============================================================
# 1. CONFIG
# ============================================================

INPUT_FULL_PATH = "data_cleaned_full.csv"

GOLD_CANDIDATES = [
    "human_gold_500_cleaned_final.csv",
    "human_gold_500_final.csv",
]

OUTPUT_FINAL_PATH = "data_final_labeled_ai.csv"


# ============================================================
# 2. BASIC UTILITIES
# ============================================================

VIETNAMESE_DIACRITIC_RE = re.compile(
    r"[àáạảãâầấậẩẫăằắặẳẵ"
    r"èéẹẻẽêềếệểễ"
    r"ìíịỉĩ"
    r"òóọỏõôồốộổỗơờớợởỡ"
    r"ùúụủũưừứựửữ"
    r"ỳýỵỷỹđ"
    r"ÀÁẠẢÃÂẦẤẬẨẪĂẰẮẶẲẴ"
    r"ÈÉẸẺẼÊỀẾỆỂỄ"
    r"ÌÍỊỈĨ"
    r"ÒÓỌỎÕÔỒỐỘỔỖƠỜỚỢỞỠ"
    r"ÙÚỤỦŨƯỪỨỰỬỮ"
    r"ỲÝỴỶỸĐ]"
)

WORD_CHAR_CLASS = r"\w"


def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFC", str(text))


def norm_text(text) -> str:
    if pd.isna(text):
        return ""
    text = normalize_unicode(str(text)).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_count(text) -> int:
    text = norm_text(text)
    return len(text.split()) if text else 0


def append_note(old, note: str) -> str:
    old = "" if pd.isna(old) else str(old)
    if note in old:
        return old
    return (old + note + "; ").strip()


def phrase_pattern(phrase: str) -> str:
    phrase = norm_text(phrase)
    escaped = re.escape(phrase)
    return rf"(?<!{WORD_CHAR_CLASS}){escaped}(?!{WORD_CHAR_CLASS})"


def contains_phrase(text, phrases: Iterable[str]) -> bool:
    text = norm_text(text)
    for phrase in phrases:
        phrase = norm_text(phrase)
        if not phrase:
            continue
        if re.search(phrase_pattern(phrase), text, flags=re.IGNORECASE):
            return True
    return False


def contains_regex(text, patterns: Iterable[str]) -> bool:
    text = norm_text(text)
    return any(re.search(pattern, text, flags=re.IGNORECASE) is not None for pattern in patterns)


def has_vietnamese_diacritic(text) -> bool:
    return VIETNAMESE_DIACRITIC_RE.search(str(text)) is not None


def is_likely_english(text) -> bool:
    """
    Heuristic chặt để loại comment tiếng Anh ngoài phạm vi.
    Không loại tiếng Việt không dấu nếu không có nhiều từ tiếng Anh phổ biến.
    """
    text = norm_text(text)

    if has_vietnamese_diacritic(text):
        return False

    tokens = re.findall(r"[a-zA-Z]+", text)
    if len(tokens) < 4:
        return False

    english_words = {
        "i", "you", "he", "she", "it", "we", "they",
        "the", "a", "an", "and", "or", "but", "if",
        "from", "with", "for", "to", "of", "in", "on", "at",
        "this", "that", "these", "those",
        "have", "has", "had", "got", "get", "came", "come",
        "was", "were", "is", "are", "am", "be", "been",
        "my", "your", "his", "her", "their", "our",
        "good", "bad", "better", "best", "buy", "bought",
        "phone", "car", "team", "game", "match", "player", "screen",
        "camera", "battery", "price", "black", "white", "new", "old",
    }

    english_count = sum(1 for t in tokens if t in english_words)
    ratio = english_count / max(len(tokens), 1)

    return english_count >= 2 and ratio >= 0.25

In [2]:
# ============================================================
# 3. MATCH GOLD SET WITH FULL DATASET WITHOUT USING ID
# ============================================================

def find_gold_path() -> str:
    for path in GOLD_CANDIDATES:
        if Path(path).exists():
            return path
    raise FileNotFoundError(f"Không tìm thấy gold set trong các file: {GOLD_CANDIDATES}")


def ensure_match_columns(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()

    for col in ["topic", "video_id", "source_item_id", "created_at", "text", "clean_text"]:
        if col not in frame.columns:
            frame[col] = ""

    frame["__topic_key"] = frame["topic"].apply(norm_text)
    frame["__video_id_key"] = frame["video_id"].apply(norm_text)
    frame["__source_item_id_key"] = frame["source_item_id"].apply(norm_text)
    frame["__created_at_key"] = frame["created_at"].apply(norm_text)
    frame["__text_key"] = frame["text"].apply(norm_text)
    frame["__clean_text_key"] = frame["clean_text"].apply(norm_text)

    return frame


def build_match_key(row: pd.Series, fields: List[str]) -> str:
    values = []

    for field in fields:
        value = row.get(f"__{field}_key", "")
        value = "" if pd.isna(value) else str(value)
        values.append(value)

    return "||".join(values)


def valid_match_key(key: str, fields: List[str]) -> bool:
    parts = key.split("||")

    if len(parts) != len(fields):
        return False

    for field, value in zip(fields, parts):
        if field in {"topic", "text", "clean_text"} and value == "":
            return False

    strong_fields = {"video_id", "source_item_id", "created_at"}

    for field, value in zip(fields, parts):
        if field in strong_fields and value == "":
            return False

    return True


def make_full_key_map(full_work: pd.DataFrame, fields: List[str]) -> dict:
    tmp = full_work[["__full_index"] + [f"__{field}_key" for field in fields]].copy()

    tmp["__match_key"] = tmp.apply(
        lambda row: build_match_key(row, fields),
        axis=1
    )

    tmp = tmp[tmp["__match_key"].apply(lambda x: valid_match_key(x, fields))]

    return (
        tmp.groupby("__match_key")["__full_index"]
        .apply(list)
        .to_dict()
    )


def match_gold_to_full(full: pd.DataFrame, gold: pd.DataFrame) -> pd.DataFrame:
    """
    Match gold set với full dataset bằng khóa mạnh.
    Tuyệt đối không dùng id đơn lẻ.
    """
    full_work = ensure_match_columns(full)
    gold_work = ensure_match_columns(gold)

    full_work["__full_index"] = full_work.index
    gold_work["__gold_row_index"] = gold_work.index

    match_stages = [
        ("topic+video_id+created_at+text", ["topic", "video_id", "created_at", "text"], False),
        ("topic+source_item_id+created_at+text", ["topic", "source_item_id", "created_at", "text"], False),
        ("topic+video_id+text", ["topic", "video_id", "text"], False),
        ("topic+source_item_id+text", ["topic", "source_item_id", "text"], False),
        ("topic+created_at+text", ["topic", "created_at", "text"], False),

        # Fallback nếu text trong Excel bị biến dạng, chỉ dùng khi unique.
        ("topic+video_id+created_at_unique", ["topic", "video_id", "created_at"], True),
        ("topic+source_item_id+created_at_unique", ["topic", "source_item_id", "created_at"], True),

        # Fallback cuối, chỉ dùng nếu unique trong full.
        ("topic+text_unique", ["topic", "text"], True),
        ("topic+clean_text_unique", ["topic", "clean_text"], True),
    ]

    reports = []
    matched_gold_indices = set()

    for method, fields, require_unique in match_stages:
        full_map = make_full_key_map(full_work, fields)

        for _, gold_row in gold_work.iterrows():
            gold_idx = int(gold_row["__gold_row_index"])

            if gold_idx in matched_gold_indices:
                continue

            key = build_match_key(gold_row, fields)

            if not valid_match_key(key, fields):
                continue

            candidates = full_map.get(key, [])
            if not candidates:
                continue

            unique_candidates = sorted(set(int(x) for x in candidates))

            if require_unique and len(unique_candidates) != 1:
                continue

            matched_gold_indices.add(gold_idx)

            reports.append({
                "gold_row_index": gold_idx,
                "gold_topic": gold_row.get("topic", ""),
                "gold_text": gold_row.get("text", ""),
                "gold_final_is_relevant": gold_row["final_is_relevant"],
                "gold_final_manual_label": gold_row["final_manual_label"],
                "match_method": method,
                "matched_full_count": len(unique_candidates),
                "matched_full_indices": "|".join(str(x) for x in unique_candidates),
                "matched_full_topics": "|".join(str(full.loc[x, "topic"]) for x in unique_candidates),
            })

    for _, gold_row in gold_work.iterrows():
        gold_idx = int(gold_row["__gold_row_index"])

        if gold_idx not in matched_gold_indices:
            reports.append({
                "gold_row_index": gold_idx,
                "gold_topic": gold_row.get("topic", ""),
                "gold_text": gold_row.get("text", ""),
                "gold_final_is_relevant": gold_row["final_is_relevant"],
                "gold_final_manual_label": gold_row["final_manual_label"],
                "match_method": "unmatched",
                "matched_full_count": 0,
                "matched_full_indices": "",
                "matched_full_topics": "",
            })

    report = (
        pd.DataFrame(reports)
        .sort_values("gold_row_index")
        .reset_index(drop=True)
    )

    return report

In [3]:
# ============================================================
# 4. AI-FIRST CALIBRATION RULES
# ============================================================

TOPIC_STRONG_SIGNALS = {
    "arsenal": [
        "arsenal", "ars", "asn", "pháo", "pháo thủ", "phấn", "phấn đào",
        "sơ nan", "a sơ nan", "phốn lào", "phú ngao", "a phú thảo",
        "arteta", "ateta", "aterta", "rice", "zubimendi", "saka", "havertz",
        "odegaard", "martinelli", "rung chuông", "wenger", "emirates",
    ],
    "iphone17series": [
        "iphone", "ip", "iphone 17", "17 pro", "17 pro max", "17prm",
        "17 prm", "iphone air", "ip air", "apple", "ios", "a19",
        "16 pro", "16 pro max", "16prm", "16 prm", "pro max", "promax",
        "ceramic shield", "magsafe",
    ],
    "ronaldo": [
        "ronaldo", "cr7", "r7", "cristiano", "siu", "siuu",
        "anh 7", "anh bảy", "cu đô", "al nassr", "al-nassr",
        "bồ đào nha", "portugal", "1000 bàn",
    ],
    "s25series": [
        "s25", "s25u", "s25 ultra", "s25 plus", "galaxy s25",
        "samsung", "galaxy", "one ui", "snapdragon", "exynos",
        "galaxy ai", "sạc 45w", "45w",
    ],
    "vinfast_vf3": [
        "vf3", "vf 3", "vinfast vf3", "xe vf3", "vinfast",
        "xe điện mini", "đặt cọc", "quãng đường", "cốp", "vô lăng",
        "cần số", "gạt nước", "gạt mưa", "hấp hơi", "firmware",
        "điều hòa", "lỗi xe", "bảo hành", "trạm sạc",
    ],
}

TOPIC_COMPONENT_SIGNALS = {
    "iphone17series": [
        "camera", "pin", "chip", "cpu", "màn hình", "kính", "titan",
        "sạc", "ios", "hiệu năng", "thiết kế", "bảo hành", "giảm giá",
        "xước", "pro max", "apple store",
    ],
    "s25series": [
        "camera", "pin", "sạc", "màn hình", "chip", "hiệu năng",
        "one ui", "snapdragon", "exynos", "galaxy ai", "sọc", "nóng", "45w",
    ],
    "vinfast_vf3": [
        "pin", "sạc", "cốp", "vô lăng", "cần số", "gạt", "hấp hơi",
        "firmware", "điều hòa", "lỗi", "bảo hành", "quãng đường",
        "trạm sạc", "đặt cọc", "xe điện", "xe điện mini",
    ],
}

OFF_TOPIC_ONLY = {
    "arsenal": [
        "manchester city", "man city", "mc", "mct", "pep", "haaland",
        "haland", "cherki", "semenyo", "semeyo", "man xanh", "the citizens",
    ],
    "ronaldo": [
        "messi", "fan messi", "m10", "goat messi", "pele", "mp10", "hl9",
    ],
    "s25series": [
        "iphone", "ip", "s23", "s24", "xiaomi", "oppo", "vivo",
    ],
    "vinfast_vf3": [
        "vf5", "mercedes", "xe trung quốc", "toyota", "honda",
    ],
}

IPHONE_MARKET_ONLY = [
    "ram", "ssd", "linh kiện", "chip thế giới", "tỉ giá", "tỷ giá",
    "vnd", "hàng xách tay", "giữ giá", "chưa tăng giá", "thị trường",
]

AUXILIARY_OR_CREATOR_TERMS = [
    "video", "clip", "reviewer", "ad", "chủ kênh", "kênh",
    "blv", "bình luận viên", "dán phim", "camera hành trình",
    "shop", "link", "mua ở đâu", "xin địa chỉ",
]

SPAM_SALES_TERMS = [
    "inbox", "ib", "shop", "link mua", "đặt hàng", "giá tốt inbox",
    "zalo", "sdt", "số điện thoại", "khuyến mãi", "mã giảm giá",
]

STRONG_NEGATIVE = [
    "không đáng mua", "khỏi mua", "quá tệ", "rất tệ", "tệ nhất",
    "thất bại", "thất vọng", "quá thất vọng", "xàm", "ngu", "cùi",
    "phế", "nát", "lùa gà", "hút máu", "pin yếu", "sọc màn",
    "camera cùi", "cam cùi", "lỗi nặng", "không ổn", "đá chả ra",
    "trắng tay", "bay khỏi", "loại khỏi", "keo kiệt", "dỏm",
    "dởm", "rác", "kém hơn", "giảm chất lượng", "của nợ",
    "phá sản", "khuyên bán", "đồ chơi", "chỉ chạy trong phố",
]

SARCASM_NEGATIVE = [
    "phốn lào", "phú ngao", "kẹt pháo", "bàn danh dự", "hạng nhì",
    "tứ quý 2", "trắng tay", "lời nguyền", "rung chuông",
    "cà khịa", "cười như được mùa", "đội đua vô địch",
    "chả làm được", "chả làm đc", "đéo thèm xem",
]

STRONG_POSITIVE = [
    "quá đỉnh", "rất đỉnh", "đỉnh cao", "đẳng cấp", "vĩ đại",
    "quá ngon", "rất ngon", "đáng mua", "quá tốt", "rất tốt",
    "tuyệt vời", "quá xịn", "ổn áp", "quá ổn", "quá thơm",
    "chúc mừng", "tự hào", "rất thích", "siuu", "quá hay",
    "hay quá", "xứng đáng", "đáng tiền", "pin tốt", "camera đẹp",
    "màn đẹp", "dùng tốt", "xài ổn",
]

INFO_QUESTION_PATTERNS = [
    r"\?",
    r"bao\s+nhiêu",
    r"giá\s+(bao\s+nhiêu|nhiêu|bn|hiện\s+tại)",
    r"mua\s+ở\s+đâu",
    r"có\s+nên",
    r"nên\s+mua",
    r"khi\s+nào",
    r"so\s+với",
    r"như\s+nào",
    r"thế\s+nào",
    r"pin\s+(ổn|tốt|oke)\s+(không|ko|k|kh)",
    r"(ổn|tốt|oke|đẹp|hay)\s+(không|ko|k|kh|nhỉ|hả)",
    r"\b[\w\d]+\s+hay\s+[\w\d]+\b",
]

NEGATED_NEGATIVE_PATTERNS = [
    r"không\s+(bị\s+)?(lỗi|nóng|sọc|lag|đơ|yếu|xấu|tệ|chán|cùi)",
    r"chưa\s+(bị\s+)?(lỗi|nóng|sọc|lag|đơ|yếu|xấu|tệ|chán|cùi)",
    r"chưa\s+gặp\s+(lỗi|sọc|vấn đề)",
    r"không\s+gặp\s+(lỗi|sọc|vấn đề)",
    r"không\s+có\s+(lỗi|vấn đề)",
]

RHETORICAL_NEGATIVE_PATTERNS = [
    r"thua\s+là\s+thua",
    r"hỏi\s+vì\s+sao",
    r"ai\s+mà\s+mua",
    r"có\s+ai\s+mua\s+đâu",
    r"ngu\s+gì\s+mua",
    r"mua\s+cái\s+niềm\s+tin",
    r"phản\s+biện\s+đi",
]


def has_topic_signal(row) -> bool:
    topic = str(row["topic"])
    text = row["clean_text_norm"]
    return contains_phrase(text, TOPIC_STRONG_SIGNALS.get(topic, []))


def has_component_signal(row) -> bool:
    topic = str(row["topic"])
    text = row["clean_text_norm"]
    return contains_phrase(text, TOPIC_COMPONENT_SIGNALS.get(topic, []))


def is_offtopic_only(row) -> bool:
    topic = str(row["topic"])
    text = row["clean_text_norm"]
    off_terms = OFF_TOPIC_ONLY.get(topic, [])

    if not off_terms:
        return False

    return contains_phrase(text, off_terms) and not has_topic_signal(row) and not has_component_signal(row)


def is_market_only_iphone(row) -> bool:
    if str(row["topic"]) != "iphone17series":
        return False

    text = row["clean_text_norm"]

    return (
        contains_phrase(text, IPHONE_MARKET_ONLY)
        and not has_topic_signal(row)
        and not has_component_signal(row)
    )


def is_auxiliary_only(row) -> bool:
    text = row["clean_text_norm"]

    return (
        contains_phrase(text, AUXILIARY_OR_CREATOR_TERMS)
        and not has_topic_signal(row)
        and not has_component_signal(row)
    )


def is_spam_sales(text) -> bool:
    return contains_phrase(text, SPAM_SALES_TERMS) or contains_regex(text, [r"http[s]?://", r"www\."])


def is_too_short_without_signal(row) -> bool:
    return (
        token_count(row["clean_text_norm"]) <= 2
        and not has_topic_signal(row)
        and not has_component_signal(row)
    )


def has_clear_negative(text) -> bool:
    text = norm_text(text)

    if contains_regex(text, NEGATED_NEGATIVE_PATTERNS):
        return False

    return (
        contains_phrase(text, STRONG_NEGATIVE)
        or contains_phrase(text, SARCASM_NEGATIVE)
        or contains_regex(text, RHETORICAL_NEGATIVE_PATTERNS)
    )


def has_clear_positive(text) -> bool:
    text = norm_text(text)

    if contains_phrase(text, STRONG_NEGATIVE + SARCASM_NEGATIVE):
        return False

    return contains_phrase(text, STRONG_POSITIVE)


def is_info_question(text) -> bool:
    text = norm_text(text)

    if has_clear_negative(text) or has_clear_positive(text):
        return False

    return contains_regex(text, INFO_QUESTION_PATTERNS)


def is_creator_or_auxiliary_sentiment(row) -> bool:
    text = row["clean_text_norm"]

    return (
        contains_phrase(text, AUXILIARY_OR_CREATOR_TERMS)
        and not has_clear_negative(text)
        and not has_clear_positive(text)
    )


def is_mixed_without_clear_direction(text) -> bool:
    text = norm_text(text)

    has_pos = contains_phrase(text, STRONG_POSITIVE)
    has_neg = contains_phrase(text, STRONG_NEGATIVE)

    return has_pos and has_neg and not contains_phrase(text, SARCASM_NEGATIVE)

In [4]:
# ============================================================
# 5. APPLY CALIBRATION RULES
# ============================================================

def set_relevance(df, mask, value, note):
    mask = mask.fillna(False)

    old_rel = df.loc[mask, "final_is_relevant"]
    df.loc[mask, "final_is_relevant"] = value

    # Nếu irrelevant thì sentiment bắt buộc neutral.
    if value == 0:
        df.loc[mask, "final_manual_label"] = 2

    changed = mask & (old_rel != value)
    df.loc[changed, "label_source"] = "relevance_rule_corrected"
    df.loc[changed, "quality_flag"] = "rule_adjusted"
    df.loc[changed, "correction_note"] = df.loc[changed, "correction_note"].apply(lambda x: append_note(x, note))


def set_sentiment(df, mask, label, note):
    mask = mask.fillna(False) & (df["final_is_relevant"] == 1)

    old_label = df.loc[mask, "final_manual_label"]
    df.loc[mask, "final_manual_label"] = label

    changed = mask & (old_label != label)
    df.loc[changed, "label_source"] = "sentiment_rule_corrected"
    df.loc[changed, "quality_flag"] = "rule_adjusted"
    df.loc[changed, "correction_note"] = df.loc[changed, "correction_note"].apply(lambda x: append_note(x, note))


def mark_low_confidence(df, mask, note):
    mask = mask.fillna(False) & (df["quality_flag"] != "reviewed")

    df.loc[mask, "quality_flag"] = "low_confidence"
    df.loc[mask, "correction_note"] = df.loc[mask, "correction_note"].apply(lambda x: append_note(x, note))


def apply_relevance_calibration(df: pd.DataFrame) -> pd.DataFrame:
    """
    AI-first relevance calibration:
    - Chỉ giảm false positive rõ.
    - Khôi phục false negative có topic signal rõ.
    """
    # False positive: AI giữ nhầm comment không liên quan.
    mask = df.apply(is_too_short_without_signal, axis=1)
    set_relevance(df, mask, 0, "R1: comment quá ngắn/chung chung, thiếu ngữ cảnh")

    mask = df.apply(is_offtopic_only, axis=1)
    set_relevance(df, mask, 0, "R2: chỉ nhắc đối tượng/sản phẩm khác, thiếu tín hiệu topic chính")

    mask = df.apply(is_market_only_iphone, axis=1)
    set_relevance(df, mask, 0, "R3: thông tin giá/thị trường/linh kiện chung, thiếu trọng tâm iPhone")

    mask = df.apply(is_auxiliary_only, axis=1)
    set_relevance(df, mask, 0, "R4: nội dung hướng vào video/reviewer/AD/nhu cầu phụ trợ")

    mask = df["clean_text_norm"].apply(is_spam_sales)
    set_relevance(df, mask, 0, "R5: quảng cáo/link/shop/mua bán không phải đánh giá topic")

    mask = df["clean_text_norm"].apply(is_likely_english)
    set_relevance(df, mask, 0, "R6: comment tiếng Anh/ngoại ngữ ngoài phạm vi")

    # False negative: AI loại nhầm comment có dấu hiệu topic rõ.
    mask = df.apply(has_topic_signal, axis=1)
    set_relevance(df, mask, 1, "R7: khôi phục relevant do có alias/từ khóa topic rõ")

    mask = df.apply(has_component_signal, axis=1) & (~df["clean_text_norm"].apply(is_likely_english))
    set_relevance(df, mask, 1, "R8: khôi phục relevant do nhắc bộ phận/tính năng/trải nghiệm topic")

    # Khóa lại English sau khi recover.
    mask = df["clean_text_norm"].apply(is_likely_english)
    set_relevance(df, mask, 0, "R9: khóa loại English sau rule khôi phục")

    df.loc[df["final_is_relevant"] == 0, "final_manual_label"] = 2

    return df


def apply_sentiment_calibration(df: pd.DataFrame) -> pd.DataFrame:
    """
    AI-first sentiment calibration:
    - AI sentiment khá tốt nên chỉ sửa cue mạnh/rõ.
    - Mơ hồ thì đưa neutral hoặc low_confidence, không ép nhãn rộng.
    """
    relevant_mask = df["final_is_relevant"] == 1
    text = df["clean_text_norm"]

    # Neutral -> negative hoặc positive khi có cue rất rõ.
    mask = relevant_mask & text.apply(has_clear_negative)
    set_sentiment(df, mask, 1, "S1: mỉa mai/chê/lỗi/trải nghiệm tiêu cực rõ")

    mask = relevant_mask & text.apply(has_clear_positive)
    set_sentiment(df, mask, 0, "S2: khen/ủng hộ rõ")

    # AI diễn giải quá mức: câu hỏi/thông tin/lựa chọn -> neutral.
    mask = relevant_mask & text.apply(is_info_question)
    set_sentiment(df, mask, 2, "S3: câu hỏi/thông tin/lựa chọn, không có cảm xúc rõ")

    # Cảm xúc hướng vào video/reviewer/AD/BLV hoặc nhu cầu phụ trợ -> neutral/low_confidence.
    mask = relevant_mask & df.apply(is_creator_or_auxiliary_sentiment, axis=1)
    set_sentiment(df, mask, 2, "S4: cảm xúc hướng vào video/reviewer/AD/nhu cầu phụ trợ")
    mark_low_confidence(df, mask, "S4: có thể không hướng trực tiếp vào topic")

    # Vừa khen vừa chê cân bằng -> neutral.
    mask = relevant_mask & text.apply(is_mixed_without_clear_direction)
    set_sentiment(df, mask, 2, "S5: vừa có cue khen vừa cue chê, không nghiêng rõ")

    # Đánh dấu các dòng rủi ro, không nhất thiết đổi nhãn.
    mask = (
        relevant_mask
        & (df["clean_text_norm"].apply(token_count) <= 3)
        & (~df.apply(has_topic_signal, axis=1))
    )
    mark_low_confidence(df, mask, "S6: comment ngắn, thiếu ngữ cảnh")

    df.loc[df["final_is_relevant"] == 0, "final_manual_label"] = 2

    return df

In [5]:
# ============================================================
# 6. GOLD OVERRIDE, SCORING, SUMMARY
# ============================================================

def override_gold_set(df: pd.DataFrame, gold: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    match_report = match_gold_to_full(df, gold)

    matched = match_report[match_report["matched_full_count"] > 0].copy()

    print("Số gold rows:", len(gold))
    print("Số gold rows match được:", len(matched))
    print("Gold row matched rate:", round(len(matched) / len(gold), 4))

    print("\nGold rows match theo topic:")
    print(matched["gold_topic"].value_counts().sort_index())

    if len(matched) != len(gold):
        unmatched = match_report[match_report["matched_full_count"] == 0]
        print("\nCòn gold rows chưa match được:")
        print(unmatched[["gold_row_index", "gold_topic", "gold_text"]].head(20))
        raise ValueError("Gold set chưa match đủ. Cần kiểm tra lại trước khi tạo final dataset.")

    df["is_human_gold"] = False

    for _, row in matched.iterrows():
        full_indices = [
            int(x)
            for x in str(row["matched_full_indices"]).split("|")
            if str(x).strip() != ""
        ]

        for idx in full_indices:
            df.at[idx, "final_is_relevant"] = row["gold_final_is_relevant"]
            df.at[idx, "final_manual_label"] = row["gold_final_manual_label"]

            if df.at[idx, "final_is_relevant"] == 0:
                df.at[idx, "final_manual_label"] = 2

            df.at[idx, "label_source"] = "human_gold"
            df.at[idx, "quality_flag"] = "reviewed"
            df.at[idx, "is_human_gold"] = True
            df.at[idx, "correction_note"] = append_note(
                df.at[idx, "correction_note"],
                "Override bằng human gold set"
            )

    print("\nFull rows được override bởi gold:", int(df["is_human_gold"].sum()))
    print("Full rows human_gold theo topic:")
    print(df[df["is_human_gold"]]["topic"].value_counts().sort_index())

    # Validation gold sau override.
    relevance_matches = []
    sentiment_matches = []

    for _, row in matched.iterrows():
        full_indices = [
            int(x)
            for x in str(row["matched_full_indices"]).split("|")
            if str(x).strip() != ""
        ]

        gold_rel = row["gold_final_is_relevant"]
        gold_sent = row["gold_final_manual_label"]

        rel_match = all(df.loc[idx, "final_is_relevant"] == gold_rel for idx in full_indices)
        sent_match = all(df.loc[idx, "final_manual_label"] == gold_sent for idx in full_indices)

        relevance_matches.append(rel_match)
        sentiment_matches.append(sent_match)

    print("\nGold set relevance match:", sum(relevance_matches) / len(relevance_matches))
    print("Gold set sentiment match:", sum(sentiment_matches) / len(sentiment_matches))

    return df, match_report


def add_scores(df: pd.DataFrame) -> pd.DataFrame:
    df.loc[df["final_is_relevant"] == 0, "final_manual_label"] = 2

    score_map = {
        0: 1,
        1: -1,
        2: 0,
    }

    df["sentiment_score_raw"] = df["final_manual_label"].map(score_map)

    confidence_map = {
        "reviewed": 1.00,
        "ok": 0.85,
        "rule_adjusted": 0.75,
        "low_confidence": 0.55,
    }

    df["label_confidence"] = df["quality_flag"].map(confidence_map).fillna(0.75)
    df["sentiment_score_weighted"] = df["sentiment_score_raw"] * df["label_confidence"]

    return df



def print_calibration_report(df: pd.DataFrame) -> None:
    """
    In trực tiếp các bảng so sánh trước/sau calibration.
    Không xuất file summary riêng để notebook gọn hơn.
    """
    total_rows = len(df)
    changed_relevance = int((df["ai_is_relevant"] != df["final_is_relevant"]).sum())
    changed_manual_label = int((df["ai_manual_label"] != df["final_manual_label"]).sum())
    invalid = int(((df["final_is_relevant"] == 0) & (df["final_manual_label"] != 2)).sum())

    print("\n========== AFTER CALIBRATION ==========")

    print("Final is_relevant:")
    print(df["final_is_relevant"].value_counts().sort_index())

    print("\nFinal manual_label:")
    print(df["final_manual_label"].value_counts().sort_index())

    print("\nLabel source:")
    print(df["label_source"].value_counts())

    print("\nQuality flag:")
    print(df["quality_flag"].value_counts())

    print("\nMean label_confidence:", round(float(df["label_confidence"].mean()), 4))

    print("\n========== BEFORE vs AFTER - RELEVANCE ==========")
    rel_table = pd.crosstab(
        df["ai_is_relevant"],
        df["final_is_relevant"],
        rownames=["AI is_relevant"],
        colnames=["Final is_relevant"]
    )
    print(rel_table)

    print("\nRelevance changes:")
    print(f"AI irrelevant -> Final relevant: {int(((df['ai_is_relevant'] == 0) & (df['final_is_relevant'] == 1)).sum())}")
    print(f"AI relevant -> Final irrelevant: {int(((df['ai_is_relevant'] == 1) & (df['final_is_relevant'] == 0)).sum())}")
    print(f"Total changed relevance: {changed_relevance} / {total_rows} ({changed_relevance / total_rows:.2%})")

    print("\n========== BEFORE vs AFTER - MANUAL_LABEL ==========")
    sent_table = pd.crosstab(
        df["ai_manual_label"],
        df["final_manual_label"],
        rownames=["AI manual_label"],
        colnames=["Final manual_label"]
    )
    print(sent_table)
    print(f"\nTotal changed manual_label: {changed_manual_label} / {total_rows} ({changed_manual_label / total_rows:.2%})")

    print("\n========== FINAL RELEVANCE BY TOPIC ==========")
    print(pd.crosstab(df["topic"], df["final_is_relevant"]))

    print("\n========== FINAL MANUAL_LABEL BY TOPIC ==========")
    print(pd.crosstab(df["topic"], df["final_manual_label"]))

    relevant_df = df[df["final_is_relevant"] == 1].copy()
    print("\n========== SENTIMENT ON RELEVANT COMMENTS ONLY ==========")
    print("Số dòng relevant:", len(relevant_df))
    print("\nPhân phối sentiment trên relevant:")
    print(relevant_df["final_manual_label"].value_counts().sort_index())

    topic_sentiment = relevant_df.groupby("topic").agg(
        total_relevant=("id", "count"),
        positive_count=("final_manual_label", lambda x: (x == 0).sum()),
        negative_count=("final_manual_label", lambda x: (x == 1).sum()),
        neutral_count=("final_manual_label", lambda x: (x == 2).sum()),
        sentiment_score_mean=("sentiment_score_raw", "mean"),
        sentiment_score_weighted_mean=("sentiment_score_weighted", "mean"),
        label_confidence_mean=("label_confidence", "mean"),
    ).reset_index()

    topic_sentiment["positive_ratio"] = topic_sentiment["positive_count"] / topic_sentiment["total_relevant"]
    topic_sentiment["negative_ratio"] = topic_sentiment["negative_count"] / topic_sentiment["total_relevant"]
    topic_sentiment["neutral_ratio"] = topic_sentiment["neutral_count"] / topic_sentiment["total_relevant"]
    topic_sentiment["net_sentiment_ratio"] = (
        topic_sentiment["positive_count"] - topic_sentiment["negative_count"]
    ) / topic_sentiment["total_relevant"]

    topic_sentiment = topic_sentiment.sort_values("sentiment_score_weighted_mean", ascending=False)

    print("\nTopic sentiment summary trên relevant comments:")
    print(topic_sentiment.to_string(index=False))

    print("\n========== QUALITY CHECK ==========")
    print("Invalid final_is_relevant = 0 nhưng final_manual_label != 2:", invalid)
    print("Human gold rows:", int(df["is_human_gold"].sum()))


In [6]:
# ============================================================
# 7. MAIN PIPELINE
# ============================================================

full_path = Path(INPUT_FULL_PATH)

if not full_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {INPUT_FULL_PATH}")

gold_path = Path(find_gold_path())

full = pd.read_csv(full_path)
gold = pd.read_csv(gold_path)

print("Full dataset:", full_path)
print("Full shape:", full.shape)
print("Gold set:", gold_path)
print("Gold shape:", gold.shape)

required_full_cols = ["topic", "text", "clean_text", "is_relevant", "manual_label"]
required_gold_cols = ["topic", "text", "final_is_relevant", "final_manual_label"]

for col in required_full_cols:
    if col not in full.columns:
        raise ValueError(f"Full dataset thiếu cột bắt buộc: {col}")

for col in required_gold_cols:
    if col not in gold.columns:
        raise ValueError(f"Gold set thiếu cột bắt buộc: {col}")

# Không chuẩn hóa label rườm rà: dataset hiện đã có giá trị 0/1/2.
# Chỉ giữ lại AI label gốc để đối chiếu.
df = full.copy()

df["clean_text"] = df["clean_text"].fillna("").astype(str)
df["clean_text_norm"] = df["clean_text"].apply(norm_text)

df["ai_is_relevant"] = df["is_relevant"]
df["ai_manual_label"] = df["manual_label"]

df["final_is_relevant"] = df["ai_is_relevant"]
df["final_manual_label"] = df["ai_manual_label"]

df["label_source"] = "ai_kept"
df["quality_flag"] = "ok"
df["correction_note"] = ""

# Đảm bảo logic cơ bản.
df.loc[df["final_is_relevant"] == 0, "final_manual_label"] = 2

print("\n========== BEFORE CALIBRATION ==========")
print("AI is_relevant:")
print(df["ai_is_relevant"].value_counts().sort_index())

print("\nAI manual_label:")
print(df["ai_manual_label"].value_counts().sort_index())

# Apply AI-first calibration.
df = apply_relevance_calibration(df)
df = apply_sentiment_calibration(df)

# Override human gold set sau cùng.
df, gold_match_report = override_gold_set(df, gold)

# Add scores.
df = add_scores(df)

# Drop helper column.
df = df.drop(columns=["clean_text_norm"])

# Print calibration report directly in notebook output.
print_calibration_report(df)

# Save only final dataset. No separate summary file.
# Bỏ các cột dư khỏi file output:
# - is_relevant: trùng ý nghĩa với ai_is_relevant
# - is_human_gold: chỉ dùng nội bộ để kiểm tra override gold set
drop_output_cols = [
    "is_relevant",
    "is_relavant",   # phòng trường hợp có typo trong file cũ
    "is_human_gold",
]

output_df = df.drop(columns=drop_output_cols, errors="ignore").copy()

print("\nCác cột bị bỏ khỏi file output:")
print([col for col in drop_output_cols if col in df.columns])

print("\nSố cột trước khi lưu:", len(df.columns))
print("Số cột sau khi bỏ cột dư:", len(output_df.columns))

output_df.to_csv(OUTPUT_FINAL_PATH, index=False, encoding="utf-8-sig")

print(f"\nĐã lưu final dataset: {OUTPUT_FINAL_PATH}")

Full dataset: data_cleaned_full.csv
Full shape: (10718, 14)
Gold set: human_gold_500_cleaned_final.csv
Gold shape: (500, 12)

========== BEFORE CALIBRATION ==========
AI is_relevant:
ai_is_relevant
0    2393
1    8325
Name: count, dtype: int64

AI manual_label:
ai_manual_label
0    2313
1    2563
2    5842
Name: count, dtype: int64
Số gold rows: 500
Số gold rows match được: 500
Gold row matched rate: 1.0

Gold rows match theo topic:
gold_topic
arsenal           100
iphone17series    100
ronaldo           100
s25series         100
vinfast_vf3       100
Name: count, dtype: int64

Full rows được override bởi gold: 506
Full rows human_gold theo topic:
topic
arsenal           100
iphone17series    100
ronaldo           100
s25series         100
vinfast_vf3       106
Name: count, dtype: int64

Gold set relevance match: 1.0
Gold set sentiment match: 1.0

========== AFTER CALIBRATION ==========
Final is_relevant:
final_is_relevant
0    2836
1    7882
Name: count, dtype: int64

Final manual_lab